In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

In [10]:
# --- Config ---
PROJECT_ROOT = Path("/home/jouell/code/jouell3/plant_detect")
IMG_SIZE = 224        # EfficientNet expects 224×224
BATCH_SIZE = 32
EPOCHS_FROZEN = 10    # train head only
EPOCHS_FINE = 20      # fine-tune top layers

# --- Load labels, extract species from folder name ---
labels = pd.read_csv(PROJECT_ROOT / "data/labels.csv")
labels["species"] = labels["filename"].apply(lambda f: Path(f).parts[2])  # data/raw/<species>/...
print(labels["species"].value_counts())

species
rosemary     173
parsley      122
thyme        121
tarragon     114
mint          90
savory        81
chives        75
oregano       65
dill          57
coriander     44
Name: count, dtype: int64


In [12]:
# --- Build tf.data pipeline (loads & resizes on-the-fly, no RAM explosion) ---
le = LabelEncoder()
labels["label_enc"] = le.fit_transform(labels["species"])
NUM_CLASSES = len(le.classes_)

labels["full_path"] = labels["filename"].apply(
    lambda f: str(PROJECT_ROOT / f)
)

train_df, test_df = train_test_split(
    labels, test_size=0.15, stratify=labels["species"], random_state=42
)
train_df, val_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df["species"], random_state=42
)

def make_dataset(df, augment=False):
    paths = df["full_path"].tolist()
    labs = df["label_enc"].tolist()

    ds = tf.data.Dataset.from_tensor_slices((paths, labs))

    def load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        ds = ds.map(
            lambda x, y: (
                tf.image.random_flip_left_right(
                    tf.image.random_brightness(
                        tf.image.random_contrast(x, 0.8, 1.2),
                        0.1
                    )
                ),
                y,
            ),
            num_parallel_calls=tf.data.AUTOTUNE,
        )

    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, augment=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

In [13]:
# --- Phase 1: train classification head, backbone frozen ---
base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

model = keras.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FROZEN)


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 19s 599ms/step - accuracy: 0.1853 - loss: 2.2808 - val_accuracy: 0.1833 - val_loss: 2.2545
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 12s 570ms/step - accuracy: 0.1485 - loss: 2.2798 - val_accuracy: 0.1833 - val_loss: 2.2323
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 13s 582ms/step - accuracy: 0.1691 - loss: 2.2643 - val_accuracy: 0.1833 - val_loss: 2.2335
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 12s 563ms/step - accuracy: 0.1706 - loss: 2.2578 - val_accuracy: 0.1833 - val_loss: 2.2336
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 12s 539ms/step - accuracy: 0.1721 - loss: 2.2591 - val_accuracy: 0.1833 - val_loss: 2.2328
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 14s 621ms/step - accuracy: 0.1588 - loss: 2.2679 - val_accuracy: 0.1833 - val_loss: 2.2335
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 13s 584ms/step - accuracy: 0.1838 - loss: 2.2537 - val_accuracy: 0.1833 - val_loss: 2.2328
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 12s 555m

In [14]:
# --- Phase 2: unfreeze top 30 layers and fine-tune with low LR ---
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-4),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE,
                    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

Epoch 1/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 21s 668ms/step - accuracy: 0.1456 - loss: 2.3053 - val_accuracy: 0.1833 - val_loss: 2.2317
Epoch 2/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 15s 686ms/step - accuracy: 0.1574 - loss: 2.2712 - val_accuracy: 0.1833 - val_loss: 2.2350
Epoch 3/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 599ms/step - accuracy: 0.1691 - loss: 2.2531 - val_accuracy: 0.1833 - val_loss: 2.2373
Epoch 4/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 15s 685ms/step - accuracy: 0.1926 - loss: 2.2674 - val_accuracy: 0.1833 - val_loss: 2.2399
Epoch 5/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 13s 590ms/step - accuracy: 0.1662 - loss: 2.2552 - val_accuracy: 0.1833 - val_loss: 2.2442
Epoch 6/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 16s 713ms/step - accuracy: 0.1529 - loss: 2.2468 - val_accuracy: 0.1833 - val_loss: 2.2442
